<a href="https://colab.research.google.com/github/CPTR295/NLP-Using-Transformers/blob/main/Text_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Greedy Search Decoding

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [2]:
import pandas as pd
input_txt = 'Transformers are the'
input_ids = tokenizer(input_txt,return_tensors='pt')['input_ids'].to(device)
iterations=[]
n_steps=8
choices_per_step=5
with torch.no_grad():
  for _ in range(n_steps):
    iteration = dict()
    iteration['Input'] = tokenizer.decode(input_ids[0])
    output = model(input_ids=input_ids)
    next_token_logits = output.logits[0,-1,:]
    next_token_probs = torch.softmax(next_token_logits,dim=-1)
    sorted_ids = torch.argsort(next_token_probs,dim=-1,descending=True)
    for choice_idx in range(choices_per_step):
      token_id = sorted_ids[choice_idx]
      token_prob = next_token_probs[token_id].cpu().numpy()
      token_choice = (
          f'{tokenizer.decode(token_id)}({100*token_prob:.2f}%)'
      )
      iteration[f'Choice {choice_idx+1}'] = token_choice
    input_ids = torch.cat([input_ids,sorted_ids[None,0,None]],dim=-1)
    iterations.append(iteration)
pd.DataFrame(iterations)

,Input,Choice 1,Choice 2,Choice 3,Choice 4,Choice 5
0,Transformers are the,most(8.53%),only(4.96%),best(4.65%),Transformers(4.37%),ultimate(2.16%)
1,Transformers are the most,popular(16.78%),powerful(5.37%),common(4.96%),famous(3.72%),successful(3.20%)
2,Transformers are the most popular,toy(10.63%),toys(7.23%),Transformers(6.60%),of(5.46%),and(3.76%)
3,Transformers are the most popular toy,line(34.38%),in(18.20%),of(11.71%),brand(6.10%),line(2.69%)
4,Transformers are the most popular toy line,in(46.28%),of(15.09%),",(4.94%)",on(4.40%),ever(2.72%)
5,Transformers are the most popular toy line in,the(65.99%),history(12.42%),America(6.91%),Japan(2.44%),North(1.40%)
6,Transformers are the most popular toy line in the,world(69.26%),United(4.55%),history(4.29%),US(4.23%),U(2.30%)
7,Transformers are the most popular toy line in ...,",(39.73%)",.(30.64%),and(9.87%),with(2.32%),today(1.74%)


In [5]:
input_ids = tokenizer(input_txt,return_tensors='pt')['input_ids'].to(device)
output = model.generate(input_ids,max_new_tokens=n_steps,do_sample=True,top_k=30)
print(tokenizer.decode(output[0]))

Transformers are the only real threat to humanity, so they


In [6]:
max_length = 128
input_txt = """In a shocking finding, scientist discovered \
a herd of unicorns living in a remote, previously unexplored \
valley, in the Andes Mountains. Even more surprising to the \
researchers was the fact that the unicorns spoke perfect English.\n\n
"""
input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)
output_greedy = model.generate(input_ids, max_length=max_length,do_sample=False)
print(tokenizer.decode(output_greedy[0]))

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The researchers, from the University of California, Davis, and the University of Colorado, Boulder, were conducting a study on the Andean cloud forest, which is home to the rare species of cloud forest trees.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.


The researchers were surprised to find that the unicorns were able


Beam Search Decoding

In [7]:
import torch.nn.functional as F
def log_probs_from_logits(logits,labels):
  logp = F.log_softmax(logits,dim=-1)
  logp_label = torch.gather(logp,2,labels.unsqueeze(2)).squeeze(-1)
  return logp_label

In [11]:
def sequence_logprob(model,labels,input_len=0):
  with torch.no_grad():
    output = model(labels)
    log_probs = log_probs_from_logits(
        output.logits[:,:-1,:],labels[:,1:]
    )
    seq_log_prob = torch.sum(log_probs[:,input_len:])
  return seq_log_prob.cpu().numpy()

In [12]:
log = sequence_logprob(model,output_greedy,input_len=len(input_ids[0]))
print(tokenizer.decode(output_greedy[0]))
print(f'\nlog-prob: {log:.2f}')

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The researchers, from the University of California, Davis, and the University of Colorado, Boulder, were conducting a study on the Andean cloud forest, which is home to the rare species of cloud forest trees.


The researchers were surprised to find that the unicorns were able to communicate with each other, and even with humans.


The researchers were surprised to find that the unicorns were able

log-prob: -87.43


In [13]:
output_beam = model.generate(input_ids, max_length=max_length, num_beams=5,do_sample=False)
logp = sequence_logprob(model, output_beam, input_len=len(input_ids[0]))
print(tokenizer.decode(output_beam[0]))
print(f"\nlog-prob: {logp:.2f}")

In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.


The discovery of the unicorns was made by a team of scientists from the University of California, Santa Cruz, and the National Geographic Society.


The scientists were conducting a study of the Andes Mountains when they discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English

log-prob: -55.23
